# 05 — The Probability Simplex

See the space of *all* categorical distributions as one geometric object — a triangle for three outcomes — and learn its anatomy by hand: where the certain outcomes sit, where the uniform sits, what an edge means, why a mixture never leaves it, how a distribution *is* its own barycentric coordinates. Then draw on it the quantities of `02/01`–`02/03` — entropy as a hill, relative entropy as an asymmetric bowl, the independence surface that mutual information measures the distance to — and finally ask the question that opens the geometry arc: **how far apart are two points of this triangle?** You already own four candidate answers, and they disagree. Deciding between them is `02/06`–`02/08`.

**Prerequisites:** `02/01_shannon_entropy` (entropy, concavity, the Jensen check on random points "of the simplex", `dirichlet(1,…,1)`), `02/02_kl_divergence` (the two landscapes of $D$ on the triangle, convexity), `02/03_mutual_information` (product of marginals, $I = D(p_{XY}\|p_X\otimes p_Y)$), `01/14_trace_distance_and_fidelity` (total variation and the Bhattacharyya overlap on probability vectors), `01/12`–`01/13` (the Bloch ball: density matrices as a convex body, the diagonal slice).

**What you'll be able to do**
- Define the simplex $\Delta_{n-1}$ by its two constraints, compute its dimension, and prove it is convex — a mixture of distributions is a distribution — so the concavity of `02/01` and the convexity of `02/02` are statements *about a convex set*.
- Map a three-outcome distribution to a point of the triangle by hand (barycentric coordinates: the weights *are* the probabilities), read the anatomy — vertices, edges, centre — and see that the uniform is the centre of mass and `dirichlet(1,1,1)` is the uniform law on the triangle.
- Draw entropy as a concave hill and relative entropy as an asymmetric bowl on the triangle, and place the two-bit tables of `01/16` in the tetrahedron of joint tables, where the product tables form a curved *independence surface*.
- Draw the level sets of four candidate distances around one point — Euclidean, total variation, Hellinger, KL — and see them agree at the centre and disagree near the boundary, which is the question the rest of the arc answers.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt

from qot_course import viz
from qot_course.colors import COLORS
from qot_course.infotheory.classical import kl_divergence, mutual_information, shannon_entropy

rng = np.random.default_rng(0)   # random points of the simplex (section 2) and random mixtures (section 1)
viz.use_course_style()

## 1. The set, its dimension, and why it is convex

A distribution on $n$ outcomes is a vector $p = (p_1, \dots, p_n)$ with two constraints: $p_i \ge 0$ for every $i$, and $\sum_i p_i = 1$. The set of all of them is the **probability simplex**

$$
\Delta_{n-1} = \Big\{\,p \in \mathbb R^n : p_i \ge 0,\ \textstyle\sum_i p_i = 1\,\Big\} .
$$

Count the freedom: $n$ numbers, one linear equation, so $n - 1$ dimensions — the subscript. For $n = 2$ a segment (the coins of `01/01`, $p = (p, 1-p)$, one number); for $n = 3$ a **triangle**; for $n = 4$ a tetrahedron; in general a polytope with $n$ **vertices** $\delta_1 = (1, 0, \dots, 0), \dots, \delta_n$ — the certain outcomes.

**Convex, by hand.** Take two points $p, q \in \Delta_{n-1}$ and $\lambda \in [0, 1]$. The mixture $r = \lambda p + (1-\lambda) q$ has $r_i = \lambda p_i + (1-\lambda)q_i \ge 0$ (a sum of non-negative terms) and $\sum_i r_i = \lambda\cdot 1 + (1-\lambda)\cdot 1 = 1$. So $r \in \Delta_{n-1}$: *the segment between two distributions is made of distributions* — the simplex is a **convex set**, and "mixing" is moving along a straight line inside it. That is the set on which `02/01` proved entropy concave (a hill with no dips along any segment) and `02/02` proved relative entropy jointly convex.

**Barycentric coordinates.** Every $p$ is a mixture of the vertices — $p = \sum_i p_i\,\delta_i$, with *its own entries as the weights*, since $\sum_i p_i = 1$. So to draw the triangle, choose three points $v_1, v_2, v_3$ in the plane for the vertices and send $p \mapsto \sum_i p_i v_i$: the probabilities become the **barycentric coordinates** of the point. Put a mass $p_i$ at each corner and the point is their centre of mass. The course helper `viz.plot_simplex_points` does exactly that with an equilateral triangle; let's do it by hand first, check the helper agrees, and verify convexity on random mixtures.


In [ ]:
# the three vertices of an equilateral triangle in the plane, and the barycentric map p -> sum_i p_i v_i
V = np.array([[0.0, 0.0], [1.0, 0.0], [0.5, np.sqrt(3) / 2]])
def to_plane(p):
    p = np.asarray(p, dtype=float)
    return p @ V                                   # centre of mass of the vertices with weights p

examples = {
    "uniform (1/3, 1/3, 1/3)": np.array([1/3, 1/3, 1/3]),
    "p = (0.7, 0.2, 0.1)":      np.array([0.7, 0.2, 0.1]),
    "almost certain (0.92, 0.04, 0.04)": np.array([0.92, 0.04, 0.04]),
    "on an edge (0.5, 0.5, 0)": np.array([0.5, 0.5, 0.0]),
}
for name, p in examples.items():
    print(f"{name:36s} -> plane point {np.round(to_plane(p), 4)}   (entries >= 0: {np.all(p >= 0)}, sum = {p.sum():.3f})")
print("the centre of mass of the three vertices:", np.round(V.mean(axis=0), 4), "= the uniform's point")
def draw_triangle(ax, labels=True):
    # the empty triangle, by hand: three edges and three labelled vertices
    tri = np.vstack([V, V[:1]])
    ax.plot(tri[:, 0], tri[:, 1], color=COLORS["muted"], lw=1.5, zorder=1)
    if labels:
        for (x, y), lab, dx, dy in zip(V, [r"$\delta_1=(1,0,0)$", r"$\delta_2=(0,1,0)$", r"$\delta_3=(0,0,1)$"], [-0.07, 0.07, 0.0], [-0.045, -0.045, 0.045]):
            ax.text(x + dx, y + dy, lab, ha="center", va="center", fontsize=11, color=COLORS["text"])
    ax.set_aspect("equal"); ax.axis("off")

# convexity: 2000 random mixtures of random pairs never leave the simplex
p_r, q_r = rng.dirichlet(np.ones(3), size=2000), rng.dirichlet(np.ones(3), size=2000)
lam = rng.uniform(size=(2000, 1))
mix = lam * p_r + (1 - lam) * q_r
print(f"2000 random mixtures: min entry = {mix.min():.4f} (>= 0), max |sum - 1| = {np.max(np.abs(mix.sum(1) - 1)):.1e}")

fig, axes = plt.subplots(1, 2, figsize=(14, 6.2), gridspec_kw={"wspace": 0.15})
draw_triangle(axes[0])
for k, (name, p) in enumerate(examples.items()):
    axes[0].scatter(*to_plane(p), s=120, color=[viz.SOURCE_COLOR, viz.TARGET_COLOR, viz.FLOW_COLOR, COLORS["quantum"]][k], edgecolor="white", zorder=3)
    axes[0].annotate(name, to_plane(p), xytext=(9, 6), textcoords="offset points", fontsize=10)
axes[0].set_title("By hand: the barycentric map p -> sum_i p_i v_i", pad=26)
viz.plot_simplex_points(examples, ax=axes[1], title="The course helper, same triangle")
plt.show()

**Read the output and the figure.** The by-hand barycentric map puts the uniform at the centre of mass of the three corners, and the course helper (right) draws the same triangle with the same points — every example lands where its weights say: $(0.7, 0.2, 0.1)$ pulled toward $\delta_1$, the almost-certain one hugging that corner, $(0.5, 0.5, 0)$ exactly on the bottom edge, halfway between $\delta_1$ and $\delta_2$. Two thousand random mixtures stay inside to machine precision: the set is convex, and the straight segment between two distributions is the *mixture path* that `02/01` walked for concavity and `02/07`–`02/08` will compare with two other paths. Anatomy, in one breath: **vertices** = certain outcomes ($H = 0$), **edges and faces** = one outcome forbidden (the walls where `02/02`'s $D(p\|q)$ went to infinity), **centre** = the uniform ($H = \log_2 3$). Next we draw the landscapes on it.


## 2. Landscapes on the triangle: entropy, relative entropy, and the uniform law

The triangle is a space, so a function of $p$ is a picture. **Entropy** is a hill: $H(p) = \log_2 3$ at the centre, $0$ at the three corners, concave along every segment (`02/01` §2; exercise 2 there asked for this picture — here it is), and its **level sets** — the curves along which $H$ is constant — are the contour lines of that hill. **Relative entropy** to a fixed $p$ is the bowl of `02/02` §3, asymmetric and infinite at the edges. And the **uniform law on the triangle** is what `02/01` and `02/02` sampled from without saying why: the Dirichlet distribution with all parameters equal to $1$ has constant density on the simplex — a flat rain of points — so `rng.dirichlet(np.ones(3))` is "a random distribution, with no preference". The centre of mass of that rain is the uniform distribution. Let's draw the entropy hill with its level sets, the $D(\cdot\|p)$ bowl, and the rain, side by side.


In [ ]:
def simplex_grid(m=300):
    g = np.linspace(0.0, 1.0, m)
    P1, P2 = np.meshgrid(g, g)
    P3 = 1.0 - P1 - P2
    inside = P3 >= 0
    P = np.stack([P1, P2, P3], axis=-1)
    return P, inside

P, inside = simplex_grid()
XY = P @ V                                                   # every grid point mapped to the plane (barycentric)
with np.errstate(divide="ignore", invalid="ignore"):
    Hgrid = np.where(P > 0, -P * np.log2(P), 0.0).sum(-1)    # entropy, 0 log 0 = 0
    p0 = np.array([0.7, 0.2, 0.1])
    Dgrid = np.where(P > 0, P * np.log2(P / p0), 0.0).sum(-1)   # D(q || p0), finite everywhere (p0 > 0)
Hgrid[~inside] = np.nan; Dgrid[~inside] = np.nan

fig, axes = plt.subplots(1, 3, figsize=(18, 6.2), gridspec_kw={"wspace": 0.45})
for ax in axes:
    draw_triangle(ax)
cs = axes[0].tricontourf(XY[inside][:, 0], XY[inside][:, 1], Hgrid[inside], levels=np.linspace(0, np.log2(3), 21), cmap=viz.CMAP_PLAN)
axes[0].tricontour(XY[inside][:, 0], XY[inside][:, 1], Hgrid[inside], levels=[0.5, 1.0, 1.3, 1.5], colors=[COLORS["text"]], linewidths=0.7)
axes[0].scatter(*to_plane([1/3, 1/3, 1/3]), color=COLORS["highlight"], s=60, zorder=5)
fig.colorbar(cs, ax=axes[0], shrink=0.6, pad=0.04, location="bottom", label="H(p)  (bits)", ticks=[0, 0.5, 1.0, 1.5]); axes[0].set_title("Entropy H(p): a concave hill", pad=26)
cs = axes[1].tricontourf(XY[inside][:, 0], XY[inside][:, 1], Dgrid[inside], levels=np.linspace(0, 3.4, 35), cmap=viz.CMAP_COST)   # max log2(1/0.1) = 3.32 at the delta_3 corner
axes[1].tricontour(XY[inside][:, 0], XY[inside][:, 1], Dgrid[inside], levels=[0.1, 0.25, 0.5, 1.0, 2.0], colors=[COLORS["text"]], linewidths=0.7)
axes[1].scatter(*to_plane(p0), color=COLORS["highlight"], s=60, zorder=5)
fig.colorbar(cs, ax=axes[1], shrink=0.6, pad=0.04, location="bottom", label="D(q || p)  (bits)", ticks=[0, 1, 2, 3]); axes[1].set_title("D(q || p), p = (0.7, 0.2, 0.1): an asymmetric bowl", pad=26)
rain = rng.dirichlet(np.ones(3), size=3000)
axes[2].scatter(*(rain @ V).T, s=4, color=viz.SOURCE_COLOR, alpha=0.6)
axes[2].scatter(*to_plane(rain.mean(0)), color=COLORS["highlight"], s=80, zorder=5, label="centre of mass of the rain")
axes[2].legend(loc="lower center", bbox_to_anchor=(0.5, -0.2), frameon=False); axes[2].set_title("dirichlet(1, 1, 1): a flat rain", pad=26)
plt.show()

print(f"entropy at the centre: {shannon_entropy([1/3, 1/3, 1/3]):.4f} = log2 3 = {np.log2(3):.4f};   at a vertex: {shannon_entropy([1, 0, 0]):.4f};   at the edge midpoint (0.5, 0.5, 0): {shannon_entropy([0.5, 0.5, 0]):.4f}")
print(f"uniform rain: mean of 3000 dirichlet(1,1,1) points = {np.round(rain.mean(0), 3)} (the uniform);  fraction in each of the three corner triangles (p_i > 1/2): {np.round([(rain[:, i] > 0.5).mean() for i in range(3)], 3)}  (each has 1/4 of the area)")

**Read the figure and the output.** Left: the entropy hill, $\log_2 3 = 1.585$ bits at the rose centre, sliding to $0$ at the three corners; the level sets are rounded triangles that shrink toward the centre, and the edge midpoint $(0.5, 0.5, 0)$ sits at exactly $1$ bit — a fair coin on two outcomes, the third forbidden. Concavity is visible: along any straight line the surface never dips. Middle: the bowl $D(q\|p)$ around $p = (0.7, 0.2, 0.1)$, with its single zero at $p$ and its level sets reaching *farther toward* the nearby $\delta_1$ corner than toward the others (the bowl is shallow there: $D(\delta_1\|p) = \log_2(1/0.7) = 0.51$ bits against $3.32$ at $\delta_3$) — relative entropy is not a round bowl, and the same picture with the arguments swapped (`02/02` §3) has walls at the edges. Right: three thousand `dirichlet(1,1,1)` points fall as a flat rain — no clustering, each corner triangle (the region $p_i > \tfrac12$, one quarter of the area; the three of them leave the central inverted quarter) catching close to a quarter of the points — and their centre of mass lands on the uniform. That is what "a random distribution" meant in `02/01`'s Jensen check and `02/02`'s convexity check: uniform over *this* triangle.


## 3. Two bits, one tetrahedron, and the independence surface

`02/03` measured how far a joint table is from the product of its marginals. On the simplex that sentence is a picture. The joint tables of two bits, $p(x, y)$ with four entries, live in $\Delta_3$ — a **tetrahedron** with vertices $\delta_{00}, \delta_{01}, \delta_{10}, \delta_{11}$. Inside it, the *product* tables $p_X \otimes p_Y = (ab,\ a(1-b),\ (1-a)b,\ (1-a)(1-b))$ form a two-parameter family — a curved **surface**, the **independence surface**, cut out by one equation: $p_{00}\,p_{11} = p_{01}\,p_{10}$, the table's $2\times2$ determinant is zero — the classical twin of `01/17`'s amplitude-matrix determinant test (algebraic geometers call it the Segre quadric; further reading in the references). Mutual information is the relative entropy from a table to its own projection onto that surface: it is $0$ on the surface and grows away from it. The correlated coins of `01/16`, $\mathrm{diag}(\tfrac12, 0, 0, \tfrac12)$, sit on the edge joining $\delta_{00}$ to $\delta_{11}$ — as far from the surface as a two-bit table can be ($1$ bit, the ceiling of `02/03` §2).

A tetrahedron is hard to draw on a page, so take a **slice**: fix the two marginals to be fair coins, $p_X = p_Y = (\tfrac12, \tfrac12)$. That leaves one free number, the agreement probability $s = p(0,0) + p(1,1)$, so the slice is a segment from the anti-correlated coins ($s = 0$) through the independent table ($s = \tfrac12$, where the segment *pierces* the independence surface) to the correlated coins ($s = 1$). Let's compute $I$ along it — you know the answer from `02/03`: $1 - h(s)$ — and, for the full tetrahedron, check on random tables that the independence surface is exactly the zero set of $I$ and that the mixture of two product tables is in general *not* a product table (the surface is curved, not flat).


In [ ]:
s_grid = np.linspace(0.0, 1.0, 101)
def slice_table(s):                       # fair marginals, agreement probability s
    return np.array([[s / 2, (1 - s) / 2], [(1 - s) / 2, s / 2]])
I_slice = np.array([mutual_information(slice_table(s)) for s in s_grid])
h = lambda p: shannon_entropy([p, 1 - p])
print(f"slice with fair marginals: I(s) = 1 - h(s) to {np.max(np.abs(I_slice - (1 - np.array([h(s) for s in s_grid])))):.1e};  I = 0 only at s = 1/2 (the independence surface), 1 bit at both ends")

# the independence surface inside the tetrahedron: product tables have I = 0; random tables do not; mixtures of products are not products
prod_tables = [np.outer(rng.dirichlet([1, 1]), rng.dirichlet([1, 1])) for _ in range(500)]
rand_tables = rng.dirichlet(np.ones(4), size=500).reshape(-1, 2, 2)
print(f"500 random PRODUCT tables: max I = {max(mutual_information(T) for T in prod_tables):.1e} bits (on the surface);  max |det| = {max(abs(np.linalg.det(T)) for T in prod_tables):.1e} (the surface's equation)")
print(f"500 random tables of the tetrahedron: median I = {np.median([mutual_information(T) for T in rand_tables]):.4f} bits, max = {max(mutual_information(T) for T in rand_tables):.4f}")
T1, T2 = prod_tables[0], prod_tables[1]
print(f"mixture of two product tables: I = {mutual_information(0.5 * T1 + 0.5 * T2):.4f} bits > 0, det = {np.linalg.det(0.5 * T1 + 0.5 * T2):.4f} -- the surface is curved (a generic chord leaves it; chords between tables sharing a marginal stay on it)")
print(f"the correlated coins diag(1/2, 0, 0, 1/2): I = {mutual_information(slice_table(1.0)):.4f} bit, on the edge delta_00 -- delta_11, the farthest a two-bit table gets from independence")

fig, ax = plt.subplots(figsize=(7.5, 4.0))
ax.plot(s_grid, I_slice, color=viz.SOURCE_COLOR, lw=2.4, label="I(X;Y) along the slice")
ax.axvline(0.5, color=COLORS["muted"], ls="--", lw=1); ax.scatter([0.5], [0.0], color=COLORS["highlight"], s=60, zorder=5, label="pierces the independence surface")
ax.set_xlabel("agreement probability s   (0 = anti-correlated coins, 1/2 = independent, 1 = correlated coins)"); ax.set_ylabel("bits")
ax.set_title("A segment through the tetrahedron of two-bit tables, at fair marginals", pad=10); ax.legend(loc="upper center")
plt.show()

**Read the output and the figure.** Along the fair-marginal slice, $I = 1 - h(s)$ to machine precision — `02/03`'s U, now read as *distance from a segment to a surface*: $1$ bit at both ends (the two edges of the tetrahedron where the coins agree always or never), $0$ exactly where the segment pierces the independence surface. Five hundred product tables have $I = 0$ and determinant $0$ to rounding — they *are* the surface; random tables of the tetrahedron sit off it (median $0.07$ bits); and the midpoint of two generic product tables is *not* a product table ($I = 0.009 > 0$, determinant nonzero): the independence surface is curved, so the mixture path between two independent tables generically leaves independence (it stays on the surface only when the two tables share a marginal — the determinant equation is bilinear, so the surface is doubly ruled). Keep this picture for module 04: there, a pair of signals will be embedded as a joint *quantum* state, and the question "how correlated?" is again a distance to a product family — with the quantum mutual information of `02/10` in the role of $I$.


## 4. Four rulers on one triangle — the question of the arc

You already own four ways to say how far apart two points $p, q$ of the triangle are, and you have used each:

- **Euclidean**: $\|p - q\|_2$ — treat the triangle as a flat piece of the plane.
- **Total variation** (`01/14`): $\mathrm{TV} = \tfrac12\|p - q\|_1 = \tfrac12\sum_i|p_i - q_i|$ — the best one-shot guessing advantage.
- **Hellinger** (from `01/14`'s Bhattacharyya overlap $B = \sum_i\sqrt{p_i q_i}$): $\mathrm{Hel} = \sqrt{1 - B}$ in this course's convention (some authors use $\|\sqrt p - \sqrt q\|_2 = \sqrt{2(1 - B)}$, a factor $\sqrt2$ larger) — a distance, triangle inequality included (`02/07` shows why, through a square-root map).
- **Relative entropy** (`02/02`): $D(p\|q)$ — not a distance, asymmetric; we draw the direction $q \mapsto D(p\|q)$, *truth fixed, model varies*, the one that is infinite at every edge.

Draw the **level set** of each around the same centre $p$ — all points at a fixed "distance" from $p$ — and they are four different curves. Near the centre of the triangle, for a small radius, three of them look alike: $\|q - p\|_2^2$, $\mathrm{Hel}^2$ and $D$ are all *quadratic forms* in $q - p$ to leading order, so their level sets are small ellipses, while total variation is polyhedral — its level set is a hexagon at every scale. Near the boundary they part company, because they treat a *rare outcome* differently: Euclidean and TV do not care whether a change of $0.01$ happens to an outcome of probability $0.5$ or $0.01$; Hellinger and KL care a great deal (a relative change of $100\%$ versus $2\%$). That is the question of the arc, made visible: **which ruler is the natural one for distributions?** `02/06` answers with a criterion — distance should be *statistical distinguishability*, unchanged by relabelling or by splitting an outcome into sub-outcomes — and builds the metric that criterion forces; `02/07` walks its shortest paths; and `02/08` shows there is a *second*, entirely different answer once the outcomes have a geometry of their own. Let's draw the four level sets around two centres, one near the middle and one near an edge.


In [ ]:
def level_sets(p):
    # distances from every grid point q to p, on the simplex grid; returns dict name -> field
    with np.errstate(divide="ignore", invalid="ignore"):
        euc = np.linalg.norm(P - p, axis=-1)
        tv = 0.5 * np.abs(P - p).sum(-1)
        hel = np.sqrt(np.clip(1.0 - np.sqrt(np.clip(P, 0, None) * p).sum(-1), 0, None))
        kl = np.minimum((p * np.log2(p / P)).sum(-1), 10.0)           # D(p || q): truth p fixed, model q varies; +inf wherever q has a zero (capped at 10 bits so the contour routine accepts it)
    fields = {"Euclidean": euc, "total variation": tv, "Hellinger": hel, "KL  D(p || q)": kl}
    for k in fields:
        fields[k] = np.where(inside, fields[k], np.nan)
    return fields

centres = {"near the middle: p = (0.4, 0.35, 0.25)": np.array([0.4, 0.35, 0.25]),
           "near an edge (zoomed): p = (0.55, 0.42, 0.03)": np.array([0.55, 0.42, 0.03])}
levels = {"Euclidean": 0.15, "total variation": 0.12, "Hellinger": 0.07, "KL  D(p || q)": 0.03}   # one contour each, chosen so the curves are comparable in size
palette = {"Euclidean": COLORS["muted"], "total variation": viz.TARGET_COLOR, "Hellinger": viz.FLOW_COLOR, "KL  D(p || q)": viz.SOURCE_COLOR}

fig, axes = plt.subplots(1, 2, figsize=(14, 5.8), gridspec_kw={"wspace": 0.3})
for ax, (title, p) in zip(axes, centres.items()):
    draw_triangle(ax, labels=(ax is axes[0]))      # the zoomed panel shows only the edge
    fields = level_sets(p)
    for name, field in fields.items():
        ax.tricontour(XY[inside][:, 0], XY[inside][:, 1], field[inside], levels=[levels[name]], colors=[palette[name]], linewidths=2.2)
        ax.plot([], [], color=palette[name], lw=2.2, label=f"{name} = {levels[name]}")
    ax.scatter(*to_plane(p), color=COLORS["highlight"], s=60, zorder=5)
    ax.set_title(title, pad=26); ax.legend(loc="upper right", fontsize=9)
axes[1].set_xlim(0.3, 0.8); axes[1].set_ylim(-0.02, 0.24)        # zoom on the edge region; the triangle's bottom edge is the line y = 0
axes[1].text(0.55, -0.012, "edge q3 = 0", ha="center", va="top", color=COLORS["muted"], fontsize=10)
fig.suptitle("Four rulers, one triangle: the level sets agree near the middle and disagree near the boundary", y=1.0)
plt.show()

# how close does each level set of the right panel come to the edge q3 = 0?  (computed, not read off pixels)
p_edge_c = centres["near an edge (zoomed): p = (0.55, 0.42, 0.03)"]
q3_scan = np.linspace(1e-4, 0.03, 3000); q1_scan = np.linspace(0.3, 0.8, 2000)
def closest_q3(dist_fn, level):
    for q3 in q3_scan:
        Q = np.stack([q1_scan, 1 - q1_scan - q3, np.full_like(q1_scan, q3)], axis=1)
        if dist_fn(Q).min() <= level:
            return q3
    return np.nan
print("closest approach of each level set to the edge q3 = 0 (the centre has p3 = 0.03):")
print(f"  Euclidean {levels['Euclidean']} and TV {levels['total variation']}: cross the edge (the minimum Euclidean distance from p to the edge is {np.sqrt(1.5) * p_edge_c[2]:.4f} < 0.15)")
print(f"  Hellinger {levels['Hellinger']}: stops at q3 = {closest_q3(lambda Q: np.sqrt(1 - np.sqrt(p_edge_c * Q).sum(1)), levels['Hellinger']):.4f};   the edge itself is at Hellinger distance {np.sqrt(1 - np.sqrt(1 - p_edge_c[2])):.4f}")
print(f"  KL D(p||q) = {levels['KL  D(p || q)']}: stops at q3 = {closest_q3(lambda Q: (p_edge_c * np.log2(p_edge_c / Q)).sum(1), levels['KL  D(p || q)']):.4f};   the edge itself is at D = inf")

# the same disagreement in numbers: a shift of 0.02 of mass, applied to a common outcome vs a rare one
p = np.array([0.5, 0.48, 0.02])
q_common = np.array([0.48, 0.50, 0.02])   # move 0.02 between the two common outcomes
q_rare = np.array([0.50, 0.50, 0.00])     # move the rare outcome's 0.02 onto a common one
for name, q in [("shift 0.02 between common outcomes", q_common), ("remove the rare outcome's 0.02   ", q_rare)]:
    B = np.sqrt(p * q).sum()
    print(f"{name}:  Euclidean {np.linalg.norm(p - q):.4f}   TV {0.5 * np.abs(p - q).sum():.4f}   Hellinger {np.sqrt(1 - B):.4f}   D(p||q) {kl_divergence(p, q):.4f}")

**Read the figure and the output.** Around the centre-ish point (left) the four level sets are four loops of similar size and shape — the rulers broadly agree on what "nearby" means: three rounded and one hexagonal, as the quadratic-versus-polyhedral remark predicted, and the Hellinger and KL loops lie almost on top of each other (they are the *same* quadratic form near a point — `02/06` names it). Around the point near the bottom edge (right, zoomed) they split: the Euclidean circle and the TV hexagon run straight into the edge and are cut off by it, as if the boundary were an arbitrary line drawn across a flat plane; the Hellinger and KL curves flatten against the edge and stop short of it — by margins the figure can barely resolve, so they are *computed* in the output: the Hellinger $0.07$ contour stops at $q_3 = 0.0056$ and the KL $0.03$ contour at $q_3 = 0.0070$, while the Euclidean and TV contours cross $q_3 = 0$. The two statistical rulers differ in *how* far the edge is: in this convention the edge $q_3 = 0$ sits at Hellinger distance $\sqrt{1 - \sqrt{0.97}} = 0.123$ from $p$ — more than three times the Euclidean $0.037$, *farther* but reachable — whereas for the truth $p$ every model $q$ with $q_3 = 0$ costs infinitely many bits, $D(p\|q) = \infty$: the KL level sets squeeze toward the edge for ever without reaching it. The numbers say the same thing with one move of $0.02$ of mass: shifted between two common outcomes it costs Euclidean $0.028$, TV $0.02$, Hellinger $0.014$, $D = 0.0012$ bits; taken away from the rare outcome — the *same* $0.02$ of mass — Euclidean and TV do not change at all, Hellinger is seven times larger, and $D(p\|q)$ is **infinite**. Flat rulers count mass; statistical rulers count *surprise* — and they disagree with each other about *how* far the boundary is (Hellinger: finite but stretched; KL: infinite in this direction). The arc now has its question and its suspects, and `02/06` brings the criterion that picks the answer.


## Your turn

1. **Your own points, and the entropy hill by hand.** Add $(0.5, 0.25, 0.25)$ and $(0.1, 0.1, 0.8)$ to the examples and plot them with `viz.plot_simplex_points`; then, without the helper, compute the plane coordinates by the barycentric map and check. Walk the straight line from $(0.5, 0.25, 0.25)$ to the uniform in $50$ steps and plot $H$ along it: is it concave, and where is the maximum? Then walk from a vertex to the opposite edge's midpoint — same question.
2. **Higher simplices.** $\Delta_3$ (four outcomes) is a tetrahedron: how many vertices, edges, faces? In general, how many $k$-dimensional faces does $\Delta_{n-1}$ have (hint: choose which outcomes are *allowed*)? Check your formula by counting the faces of $\Delta_3$ on which the entropy is at most $1$ bit everywhere.
3. **The rain is not always flat.** Sample `rng.dirichlet([1, 1, 1])`, `rng.dirichlet([5, 5, 5])` and `rng.dirichlet([0.3, 0.3, 0.3])`, $3000$ points each, and plot the three rains on the triangle side by side. Describe where each concentrates and compute the mean entropy of each rain. Which of the three would be a fair "random distribution" for a Jensen check, and why?
4. **Harder: the independence surface in 3-D, and the quantum slice.** (a) Parametrise the independence surface of two bits by $(a, b)$, map it into the tetrahedron with a $4$-vertex barycentric map into $\mathbb R^3$ (vertices of a regular tetrahedron), and draw it with `matplotlib`'s 3-D axes together with the segment of section 3 — where does the segment pierce it? (b) The quantum side: the qubit state space of `01/13` is the Bloch *ball*, and the classical two-outcome simplex is the segment of diagonal states along the $z$-axis. Using `qot_course.quantum.density`, draw $100$ random qubit density matrices (random Bloch vectors inside the ball) and mark where their *diagonals* $(\rho_{00}, \rho_{11})$ fall on that segment; explain in one sentence why `01/13`'s $|+\rangle$ and $I/2$ land on the same point of the segment, and what this says about what the simplex cannot see.


## What you built

- You defined the probability simplex, counted its dimension, proved it is convex by hand (mixtures stay inside), and placed three-outcome distributions on the triangle by the barycentric map — a distribution is its own set of weights — checking the helper against your own coordinates.
- You read its anatomy and drew the landscapes of the last four notebooks on it: the entropy hill (peak $\log_2 3$ at the centre, $0$ at the corners), the asymmetric KL bowl, and the flat rain of `dirichlet(1,1,1)` whose centre of mass is the uniform.
- You placed the two-bit tables of `01/16` in their tetrahedron, saw the product tables form a curved independence surface, and re-read mutual information as the distance to it — $1 - h(s)$ along the fair-marginal segment.
- You drew four candidate distances around one point and watched them agree near the middle and disagree near the boundary — flat rulers count mass, statistical rulers count surprise — and posed the question the arc answers.

The geometry arc has its stage: a convex triangle, convex in the sense `02/01`–`02/02` already used, on which four rulers disagree exactly where probability is small. Which one is *natural* is the next notebook's business.


**Dictionary so far** (module 01's table, the four rows of the classical-information arc, plus today's row — the geometry arc begins; the quantum column is what `01/12`–`01/13` already gave you, read as geometry):

| Classical | Quantum |
|-----------|---------|
| bit ($0$ or $1$) | qubit $a_0\,|0\rangle + a_1\,|1\rangle$ (two complex amplitudes) |
| probabilities only add (non-negative) | amplitudes add *and cancel* — interference; a relative phase $e^{i\varphi}$ is physical, a global phase is not |
| a bit's state space: two isolated points | a pure qubit's state space: the Bloch **sphere** (poles = the classical bit; equator = equal superpositions) |
| outcome probabilities $p_x$ | **Born rule** $p_x = \lvert a_x\rvert^2$ — the *shadow* one measurement casts of a state; a different measurement casts a different shadow |
| overlap of two probability vectors, $\sum_x \sqrt{p_x q_x}$ (Bhattacharyya coefficient) | **inner product** $\langle\varphi|\psi\rangle$ — the same sum with amplitudes in place of $\sqrt{p_x}$; its modulus squared is the yes-probability of "are you $|\varphi\rangle$?" (fidelity, `01/14`) |
| a reversible operation on a bit is a permutation (identity, NOT); a general operation on probability vectors is a stochastic matrix, usually lossy | a **gate** is a **unitary** matrix $U$ ($U^\dagger U = I$): preserves every inner product, is undone by $U^\dagger$, and rotates the Bloch sphere rigidly (half-angle in the matrix, full angle on the sphere); the general, lossy map is the quantum channel of `01/19` |
| a quantity is a function of the bit, $f(0), f(1)$; its average is $\sum_x p_x f(x)$ | an **observable** is a Hermitian matrix $A = A^\dagger$; its average is $\langle\psi|A|\psi\rangle$ (real), the outcomes are its eigenvalues; the Paulis are gates *and* observables, and $(\langle X\rangle,\langle Y\rangle,\langle Z\rangle)$ is the Bloch vector (measured shot by shot in `01/10`) |
| there is one way to read a bit: look at it | a **measurement basis** is any orthonormal pair $\{|b_0\rangle,|b_1\rangle\}$, $P(k) = |\langle b_k|\psi\rangle|^2$; the hardware natively reads only $Z$, so every other basis is reached by *rotate, then read* — $H$ for $X$, $S^\dagger$ then $H$ for $Y$ (the recombination and $Y$-type rules of `01/02`) |
| a bit has no continuous knob: you flip it or you do not | **rotation gates** $R_x, R_y, R_z$ (and phase gates $P(\lambda)$) are continuous dials — angle $\theta$ in the matrix as $\theta/2$, full $\theta$ on the sphere; any pure state is $R_z(\phi)R_y(\theta)|0\rangle$, any single-qubit gate is $e^{i\alpha}R_z R_y R_z$ (Nielsen & Chuang Thm 4.1) |
| the average $\sum_x p_x f(x)$ is estimated from $N$ samples with a standard error $\sqrt{\mathrm{Var}(f)/N}$ | the **expectation value** $\langle A\rangle = \langle\psi|A|\psi\rangle = \sum_k \lambda_k P(k)$ is the average of the eigenvalue outcomes; from $N$ shots of a $\pm1$ observable it is estimated with standard error $\sqrt{(1 - \langle A\rangle^2)/N} \le 1/\sqrt N$; three Pauli averages determine the state (tomography, `01/15`) |
| a randomized program on bits: initialize, apply logic gates in order, read the output; run it many times and estimate the output frequencies, with sampling error | a **circuit**: qubits initialized to $|0\rangle$, unitary gates in time order, measurement into classical bits; $N$ **shots** give **counts**, a sample of the Born shadow with the error bar above; the device runs a compiled **native** version ($R_z$, $\sqrt X$, $X$, one fixed entangler), equal to yours up to a global phase; a *systematic* miss that does not shrink with $N$ is the signature of noise, and needs $\rho$ (`01/12`) |
| probability vector $p = (p_0, p_1)$: non-negative entries summing to $1$; averages are $\sum_x p_x f(x)$; a fair coin is $(\tfrac12,\tfrac12)$ | **density matrix** $\rho = |\psi\rangle\langle\psi|$ (pure case): Hermitian, unit trace, positive; diagonal = the Born probabilities $p$, off-diagonal = the **coherence** $\tfrac12\sin\theta\,e^{-i\phi}$ (modulus: how much superposition; phase: the relative phase); $P(b) = \langle b|\rho|b\rangle$, $\langle A\rangle = \mathrm{tr}(\rho A)$, gates act as $U\rho U^\dagger$; $\rho = \tfrac12(I + \vec r\cdot\vec\sigma)$ packs the Bloch vector; a global phase drops out; a $\rho$ that is diagonal in a shared basis *is* a classical $p$ — and the mixtures of `01/13` are the $\rho$'s that are no vector's outer product |
| not knowing which bit you hold is *the only* kind of uncertainty: a coin $p$; a fair coin $(\tfrac12,\tfrac12)$ is as uncertain as it gets, and a mixture of coins is a coin | a **mixed state** $\rho = \sum_i p_i|\psi_i\rangle\langle\psi_i|$ is the *second* kind of uncertainty (ignorance of which state), distinct from superposition (a definite state with random outcomes); its Bloch vector is the average of the parts' — inside the sphere, $I/2$ at the centre; **purity** $\mathrm{tr}(\rho^2) = \tfrac12(1 + |\vec r|^2) \in [\tfrac12, 1]$, eigenvalues $\tfrac12(1 \pm |\vec r|)$, von Neumann entropy = entropy of the eigenvalue coin (preview of `02/01`, `02/09`); a mixture forgets how it was prepared; **the seed:** $|+\rangle$ and $I/2$ share the diagonal $(\tfrac12,\tfrac12)$ and differ only in the corners |
| distances between probability vectors: **total variation** $\tfrac12\sum_x|p_x - q_x|$ (the best one-shot guess between $p$ and $q$ succeeds with probability $\tfrac12(1 + \mathrm{TV})$) and the **Bhattacharyya overlap** $\sum_x\sqrt{p_x q_x}$ | **trace distance** $T = \tfrac12\|\rho - \sigma\|_1 = \tfrac12|\vec r - \vec s|$ for qubits — half the distance in the Bloch **ball** (surface = pure, depth = mixedness); Helstrom: best one-shot guess succeeds with probability $\tfrac12(1 + T)$, by measuring the eigenbasis of $\rho - \sigma$; **fidelity** $F = (\mathrm{tr}\sqrt{\sqrt\rho\sigma\sqrt\rho})^2$ — *this course's convention*: $F = |\langle\varphi|\psi\rangle|^2 = \tfrac12(1 + \vec r\cdot\vec s)$ for pure states, $\langle\psi|\sigma|\psi\rangle$ for one pure, (Bhattacharyya)$^2$ for diagonal states; $F_U = \sqrt F$ is the Uhlmann fidelity of the Bures distance (`02/12`); $1 - \sqrt F \le T \le \sqrt{1 - F}$ |
| estimating a probability vector from data: count frequencies, quote $\pm\sqrt{p(1-p)/N}$; a miscalibrated counter biases every frequency by the same factor, and no number of samples fixes it — you calibrate the counter | **state tomography**: estimate $\langle X\rangle, \langle Y\rangle, \langle Z\rangle$ from three circuits' counts, each $\pm\sqrt{(1 - \langle\sigma\rangle^2)/N}$, and pack $\hat\rho = \tfrac12(I + \hat{\vec r}\cdot\vec\sigma)$ (project back onto the ball if noise pushes it out); purity, fidelity and trace distance read straight off $\hat\rho$; tomography reports the state *as seen through the meter* — a symmetric misassignment $\epsilon$ shrinks $\hat{\vec r}$ by $1 - 2\epsilon$, a lopsided one also shifts it (calibrate it away — exercise 2; `05/02` meets the same floor on a QPU) — while an idling qubit's *state* really does decohere, coherence as $e^{-t/T_2}$, population as $1 - e^{-t/T_1}$ (channels, `01/19`) |
| **marginal** of a joint table $p(a,b)$: sum over the other variable, $p_A(a) = \sum_b p(a,b)$ (row sums); independent variables factor, $p = p_A \otimes p_B$, and each marginal is the factor; *correlated* variables (the two matched coins) have perfectly random marginals although the pair is not random at all | two systems live on $\mathcal H_A \otimes \mathcal H_B$ (Kronecker product, index $k = 2a + b$, left factor first); a **product state** $\rho_A \otimes \rho_B$ is the independent case; the **partial trace** $\rho_A = \mathrm{tr}_B\,\rho_{AB}$, $(\rho_A)_{aa'} = \sum_b \rho_{ab,a'b}$ (replace each $2\times2$ block by its trace), is the only map that reproduces every local statistic $\mathrm{tr}(\rho_{AB}(M \otimes I)) = \mathrm{tr}(\rho_A M)$; a diagonal $\rho_{AB}$ reduces *exactly* to the row sums; a correlated joint state has **mixed** marginals — the *third* way a state can be mixed: it is part of something larger (`01/18` makes the whole pure and the parts still mixed) |
| **conditional logic** on two bits: controlled-NOT / XOR, $(a, b) \mapsto (a, a \oplus b)$ — a reversible *permutation* of the four outcomes; on a joint probability table it permutes the four entries; two independent operations, one per bit, can never make the bits agree when they did not before | **CNOT**, the $4\times4$ permutation unitary $|0\rangle\langle0|\otimes I + |1\rangle\langle1|\otimes X$ (controlled-$U$ pattern; control = left factor in this course, Qiskit's `cx(1, 0)`); it permutes the four basis vectors and, by linearity, acts on *superpositions* of the control: $\mathrm{CNOT}\,(\alpha|0\rangle + \beta|1\rangle)|0\rangle = \alpha|00\rangle + \beta|11\rangle$ — a product in, a non-product out (amplitude-matrix determinant $\ne 0$); no $M \otimes N$ can do that; CNOT$(H \otimes I)|00\rangle$ is the Bell state, whose corners are *coherence* between $|00\rangle$ and $|11\rangle$ (a product $|{+}{+}\rangle$ has them too; the coins have none) — entanglement proper is `01/18` |
| a joint table whose whole is *certain* (one outcome, probability $1$) has certain marginals; mixed marginals force a mixed whole (the coins); the correlation of two bits is fully described by their joint table, and any table is a mixture of point masses — every classical correlation is a *shared randomness* | **entanglement**: a *pure* joint state that is not a product (amplitude-matrix $\det\Psi \ne 0$) has **mixed parts** — pure whole, mixed parts, impossible for any table; measured by the **entanglement entropy** $S(\rho_A) = -\sum_i \lambda_i\log_2\lambda_i$ (eigenvalues of the part; $= S(\rho_B)$; $1$ bit for a Bell state; invariant under local gates $U\otimes V$) — **valid only when the whole is pure**: the coins have $S(\rho_A) = 1$ bit and no entanglement; a mixed state is **separable** if it is a mixture of products, and for two qubits the **partial transpose** decides it (a negative eigenvalue = entangled: Bell $-\tfrac12$, coins none); Bell and coins share every $Z$-statistic and differ in $X$: $\langle X\otimes X\rangle = 1$ vs $0$ |
| the general evolution of a probability vector is a **stochastic matrix** (Markov kernel) $p \to Tp$: columns are probability vectors, so probabilities stay non-negative and sum to $1$; a permutation is the reversible case; noise = a coin you did not see being tossed | a **quantum channel**: the general evolution $\rho \to \sum_k K_k\rho K_k^\dagger$ (Kraus operators, $\sum_k K_k^\dagger K_k = I$) — *derived* as "couple to an environment, evolve unitarily, trace the environment out" ($K_k = \langle k|_E\,U\,|0\rangle_E$); linear, trace-preserving and **completely** positive (the transpose is positive but sends half a Bell pair to a matrix with a $-\tfrac12$ eigenvalue — not a channel); a unitary is the reversible case ($K = U$), a stochastic matrix is the diagonal case ($K_{ij} = \sqrt{T_{ij}}\,|i\rangle\langle j|$); **decoherence = entanglement with an environment, then a partial trace**: dephasing (the environment learns the bit; coherence $\times e^{-t/T_\varphi}$), amplitude damping (energy leaks out; population $\times e^{-t/T_1}$, coherence $\times e^{-t/2T_1}$), depolarizing ($\rho \to (1-p)\rho + p\,I/2$); together they give $1/T_2 = 1/2T_1 + 1/T_\varphi$, hence $T_2 \le 2T_1$ — the laws `01/15` fitted |
| **Shannon entropy** $H(p) = -\sum_x p_x\log_2 p_x$: the average *surprise* $-\log_2 p_x$ (the log is forced by additivity of independent surprises); $0$ for a certain outcome, $\log_2 n$ for the uniform (Jensen), concave; **binary entropy** $h(p)$; joint and conditional entropy with the chain rule $H(AB) = H(A) + H(B|A)$, $H(B|A) \ge 0$, hence $H(A) \le H(AB)$ — a part is never more uncertain than the whole (coins: $1 = 1$; independent: $H(AB) = H(A) + H(B)$); the floor of lossless compression (Shannon 1948) | **von Neumann entropy** $S(\rho) = -\mathrm{tr}\,\rho\log_2\rho = H(\text{eigenvalues of }\rho)$ — the Shannon entropy of the eigenvalue coin (`01/13`): $0$ for a pure state, $1$ bit for $I/2$, $h\big(\tfrac{1 + |\vec r|}{2}\big)$ for a qubit; *but* the chain rule's positivity fails: the Bell state has $S(AB) = 0 < S(A) = 1$ (`01/18`) — a negative conditional entropy, `02/11`; the quantum side is earned in `02/09` |
| **relative entropy** (Kullback–Leibler) $D(p\|q) = \sum_x p_x\log_2\frac{p_x}{q_x}$ — the *extra* bits per symbol of coding with model $q$ when the truth is $p$ (cross-entropy minus entropy); **Gibbs**: $D \ge 0$, $= 0$ iff $p = q$ (from $\ln x \le x - 1$) — which re-proves $H \le \log_2 n$ and the source-coding floor; **not** a distance: asymmetric, no triangle inequality, $+\infty$ when $q$ gives zero to something $p$ can do; additive on products, jointly convex, and **monotone under any stochastic map** $D(Tp\|Tq) \le D(p\|q)$ — processing cannot create distinguishability; $D(p\,\|\,\text{uniform}) = \log_2 n - H(p)$ | **Umegaki relative entropy** $S(\rho\|\sigma) = \mathrm{tr}\,\rho(\log_2\rho - \log_2\sigma)$ (`02/09`): equals $D$ when $\rho, \sigma$ are diagonal in one basis, $+\infty$ when the support of $\rho$ leaves that of $\sigma$, $\ge 0$ (Klein), monotone under every quantum channel (Lindblad 1975) — and it *sees the corners*: $|{+}\rangle$ vs $I/2$ have identical diagonals, classical $D = 0$, quantum $S = 1$ bit |
| **mutual information** $I(X;Y) = D(p_{XY}\,\|\,p_X \otimes p_Y) = H(X) + H(Y) - H(XY) = H(X) - H(X|Y)$ — the bits the joint saves over the product, the failure of independence; symmetric, $\ge 0$ (Gibbs), $= 0$ iff independent, sign-blind (anti-correlation counts), sees nonlinear dependence; **ceiling** $I \le \min(H(X), H(Y))$ — two bits share at most $1$ bit; $1 - h(\epsilon)$ through a symmetric flip channel; **conditional** $I(X;Y|Z)$, chain rule $I(X;YZ) = I(X;Y) + I(X;Z|Y)$, hence the **data-processing inequality** $I(X;Z) \le I(X;Y)$ on a chain $X \to Y \to Z$; estimated from counts it is biased *upward* by $\approx \frac{(|X|-1)(|Y|-1)}{2N\ln 2}$ — compare with surrogates | **quantum mutual information** $I(A{:}B) = S(A) + S(B) - S(AB) = S(\rho_{AB}\,\|\,\rho_A \otimes \rho_B)$ (`02/10`): the same formula with von Neumann entropies, $\ge 0$, $= 0$ iff product; *total* correlation, classical and quantum together — the coins give $1$ bit (as here), the Bell state $2$ bits, **above** the classical ceiling, because $S(AB) = 0 < S(A)$; monotone under local channels |
| **transfer entropy** $\mathrm{TE}_{Y\to X} = I(X_{t+1};\,Y_t\,|\,X_t) = H(X_{t+1}|X_t) - H(X_{t+1}|X_t, Y_t)$ — the conditional mutual information of `02/03` with the axes named by time: what the source's past adds to the target's future *beyond the target's own past*; **directed** (lag-1, one symbol of history here; Schreiber 2000); $1 - h(\epsilon)$ for a noisy copy; zero in the back direction where lagged mutual information is fooled by the source's autocorrelation; estimated from counts with the same upward bias, $\approx \frac{|Z|(|X|-1)(|Y|-1)}{2N\ln 2}$, and tested against **autocorrelation-preserving surrogates** (circular shifts), never against a naive shuffle for an unconditioned measure; blind to synergy (XOR), sensitive to the chosen lag | **no quantum lift in this course** — the quantum coupling measures of module 04 ($I(A{:}B)$, Bures, the QOT cost on an embedded pair) are *symmetric*; direction, when you need it, stays with the classical transfer entropy on the raw signals (directed quantum information measures exist in the literature but are not built here) |
| the **probability simplex** $\Delta_{n-1} = \{p \in \mathbb R^n : p_i \ge 0, \sum_i p_i = 1\}$ — the set of *all* distributions on $n$ outcomes, a convex polytope of dimension $n - 1$ (a triangle for $n = 3$): vertices = certain outcomes, edges/faces = a forbidden outcome, centre = uniform; every point is a mixture of the vertices with itself as the weights (barycentric coordinates), mixtures never leave it; entropy is a concave hill on it (peak $\log_2 n$ at the centre, $0$ at the vertices), relative entropy an asymmetric bowl around any point; the product tables of two variables form a curved **independence surface** inside the simplex of joint tables, and mutual information is the relative entropy to it; the *distance* between two points is not given by the set — Euclidean, total variation, Hellinger and KL disagree, most of all near the boundary (`02/06`–`02/08` decide) | the **state space** of a $d$-level quantum system: the density matrices $\rho \ge 0$, $\mathrm{tr}\rho = 1$ — a convex body of dimension $d^2 - 1$ whose *diagonal slice* in any basis is the simplex (`01/12`: a $\rho$ diagonal in a shared basis *is* a $p$); for a qubit, the Bloch ball (`01/13`) — pure states on the surface, mixtures inside, $I/2$ at the centre — and the simplex is one diameter of it; its distances restrict to the simplex's on that slice: trace distance to total variation exactly, Bures (`02/12`) to Hellinger up to a factor $\sqrt2$ |
| transport plan | quantum coupling |

## What's next

Euclidean distance treats the triangle as flat, and the right-hand panel above says it is not the natural choice: the rulers that count surprise flatten against the edges. In `02/06_fisher_rao_metric` we ask for the ruler that measures *statistical distinguishability* — how well samples tell two nearby distributions apart — and find the Fisher information metric: its coefficients blow up at the boundary, every step near a corner is expensive, and the triangle is genuinely curved (the corners themselves stay at a finite distance, as `02/07`'s closed form will show).


## References

- S.-i. Amari, *Information Geometry and Its Applications*, Applied Mathematical Sciences 194, Springer (2016) — ch. 1–2: the simplex as a manifold, mixture and exponential coordinates. doi:10.1007/978-4-431-55978-8
- F. Nielsen, *An elementary introduction to information geometry*, Entropy **22**, 1100 (2020) — the simplex, divergences and the Fisher–Rao metric in one survey. doi:10.3390/e22101100
- T. M. Cover & J. A. Thomas, *Elements of Information Theory*, 2nd ed., Wiley (2006) — §2.7 (concavity of $H$, convexity of $D$ as statements on the simplex), §11.6 (Pinsker and the relation between distances). doi:10.1002/047174882X
- D. Geiger, C. Meek & B. Sturmfels, *On the toric algebra of graphical models*, Ann. Statist. **34**, 1463–1492 (2006) — the independence model as a (Segre) variety inside the simplex. doi:10.1214/009053606000000263
- I. Bengtsson & K. Życzkowski, *Geometry of Quantum States*, 2nd ed., Cambridge University Press (2017) — ch. 1–2 (the simplex and its geometries), ch. 8 (the space of density matrices, of which the simplex is a slice). doi:10.1017/9781139207010

**Previous:** `notebooks/02_InformationAndGeometry/04_transfer_entropy.ipynb` · **Next:** `notebooks/02_InformationAndGeometry/06_fisher_rao_metric.ipynb`
